# Avaliar o Tucano 2 Instruct (0,5B e 1,5B) para comparar com o córtex do Apolo

**Rode no Colab com GPU (T4 serve).** Este caderno carrega `Polygl0t/Tucano2-qwen-0.5B-Instruct` e `Polygl0t/Tucano2-qwen-1.5B-Instruct`
(licença Apache-2.0, contexto de 4.096 tokens, vocabulário de 49.152), roda ~30 prompts fixos em português, mede latência e memória
e salva as respostas em JSON no Google Drive, para comparar depois com as respostas do córtex `medio` (~85M) do Apolo.

Fontes: https://huggingface.co/Polygl0t/Tucano2-qwen-0.5B-Instruct e https://huggingface.co/Polygl0t/Tucano2-qwen-1.5B-Instruct

Não precisa aceitar termos no Hugging Face (os modelos não são "gated").

In [ ]:
!pip -q install -U transformers accelerate

In [ ]:
import json, os, time, gc, datetime
import torch
from google.colab import drive
drive.mount('/content/drive')
PASTA = '/content/drive/MyDrive/apolo/tucano_avaliacao'
os.makedirs(PASTA, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NENHUMA (use Ambiente de execução > Alterar tipo > GPU)')

## Os ~30 prompts fixos
Não mude a lista entre execuções: ela é a régua para comparar versões do córtex do Apolo. Se adicionar, acrescente no fim.

In [ ]:
PROMPTS = [
    # saudação
    {"id": "saud_01", "cat": "saudacao", "prompt": "Oi, tudo bem?"},
    {"id": "saud_02", "cat": "saudacao", "prompt": "Bom dia! Como você está hoje?"},
    {"id": "saud_03", "cat": "saudacao", "prompt": "Boa noite, preciso de ajuda."},
    {"id": "saud_04", "cat": "saudacao", "prompt": "Valeu pela ajuda, até mais!"},
    # identidade do Apolo (o Tucano NÃO é o Apolo: serve para ver o que ele responde sem contexto)
    {"id": "id_01", "cat": "identidade", "prompt": "Quem é você?"},
    {"id": "id_02", "cat": "identidade", "prompt": "Qual é o seu nome e o que você sabe fazer?"},
    {"id": "id_03", "cat": "identidade", "prompt": "Você é o Apolo, uma memória em grafo com um córtex próprio. Apresente-se em duas frases."},
    {"id": "id_04", "cat": "identidade", "prompt": "Quem criou você?"},
    # fatos simples
    {"id": "fato_01", "cat": "fatos", "prompt": "Qual é a capital do Brasil?"},
    {"id": "fato_02", "cat": "fatos", "prompt": "Quantos dias tem uma semana?"},
    {"id": "fato_03", "cat": "fatos", "prompt": "Quem descobriu o Brasil, segundo os livros de história?"},
    {"id": "fato_04", "cat": "fatos", "prompt": "Qual é o maior planeta do sistema solar?"},
    {"id": "fato_05", "cat": "fatos", "prompt": "O que é a fotossíntese? Responda em uma frase."},
    {"id": "fato_06", "cat": "fatos", "prompt": "Quanto é 15 mais 27?"},
    # programação
    {"id": "prog_01", "cat": "programacao", "prompt": "Como criar uma lista em Python?"},
    {"id": "prog_02", "cat": "programacao", "prompt": "O que é um JOIN em SQL?"},
    {"id": "prog_03", "cat": "programacao", "prompt": "Como faço um laço for em Python que imprime os números de 1 a 5?"},
    {"id": "prog_04", "cat": "programacao", "prompt": "Qual a diferença entre uma lista e uma tupla em Python?"},
    {"id": "prog_05", "cat": "programacao", "prompt": "Como desfazer o último commit no Git sem perder as alterações?"},
    {"id": "prog_06", "cat": "programacao", "prompt": "O que faz o comando docker ps?"},
    {"id": "prog_07", "cat": "programacao", "prompt": "Escreva uma função em JavaScript que soma dois números."},
    # parafrasear
    {"id": "para_01", "cat": "parafrase", "prompt": "Reescreva de outro jeito, mantendo o sentido: \"Como eu faço para criar uma lista em Python?\""},
    {"id": "para_02", "cat": "parafrase", "prompt": "Dê 3 maneiras diferentes de perguntar: \"Que horas são?\""},
    {"id": "para_03", "cat": "parafrase", "prompt": "Parafraseie em português informal: \"Gostaria de saber como instalar o Docker no meu computador.\""},
    {"id": "para_04", "cat": "parafrase", "prompt": "Dê 3 variações curtas desta pergunta de um usuário a um assistente: \"Me explique o que é um banco de dados.\""},
    # explicar trecho de código
    {"id": "cod_01", "cat": "explicar_codigo", "prompt": "Explique em português, de forma simples, o que este código faz:\n\n```python\nfor i in range(3):\n    print(i * 2)\n```"},
    {"id": "cod_02", "cat": "explicar_codigo", "prompt": "Explique em português o que esta consulta faz:\n\n```sql\nSELECT nome, COUNT(*) FROM pedidos GROUP BY nome HAVING COUNT(*) > 1;\n```"},
    {"id": "cod_03", "cat": "explicar_codigo", "prompt": "Explique em português o que este código faz:\n\n```python\ndef dobro(xs):\n    return [x * 2 for x in xs if x > 0]\n```"},
    {"id": "cod_04", "cat": "explicar_codigo", "prompt": "Explique em português o que este comando faz:\n\n```bash\ngrep -rn \"TODO\" src | wc -l\n```"},
]
print(len(PROMPTS), 'prompts')

## Função de geração (parâmetros recomendados pelo cartão do modelo: temperatura 0,1, repetition_penalty 1,2)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

MODELOS = {
    "tucano2-0.5b-instruct": "Polygl0t/Tucano2-qwen-0.5B-Instruct",
    "tucano2-1.5b-instruct": "Polygl0t/Tucano2-qwen-1.5B-Instruct",
}
MAX_NOVOS = 200

def carregar(model_id):
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16  # T4 (7.5) não tem bf16 nativo
    tok = AutoTokenizer.from_pretrained(model_id)
    t0 = time.time()
    m = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, device_map="auto")
    m.eval()
    return tok, m, dtype, time.time() - t0

def gerar(tok, m, prompt):
    msgs = [{"role": "user", "content": prompt}]
    texto = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ent = tok(texto, return_tensors="pt").to(m.device)
    cfg = GenerationConfig(do_sample=True, temperature=0.1, top_k=50, top_p=1.0, repetition_penalty=1.2,
                           max_new_tokens=MAX_NOVOS, pad_token_id=tok.eos_token_id)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        out = m.generate(**ent, generation_config=cfg)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    dt = time.time() - t0
    novos = out[0][ent.input_ids.shape[1]:]
    resp = tok.decode(novos, skip_special_tokens=True).strip()
    return resp, dt, int(novos.shape[0]), int(ent.input_ids.shape[1])

In [ ]:
RESULTADOS = {"data": datetime.datetime.now().isoformat(timespec="seconds"), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "modelos": {}}

for nome, mid in MODELOS.items():
    print("=" * 70, "\n", nome, mid)
    torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None
    tok, m, dtype, t_carga = carregar(mid)
    # aquecimento (a primeira chamada é lenta por compilar kernels)
    gerar(tok, m, "Olá")
    itens = []
    for p in PROMPTS:
        resp, dt, n_novos, n_ent = gerar(tok, m, p["prompt"])
        itens.append({**p, "resposta": resp, "latencia_s": round(dt, 3), "tokens_gerados": n_novos,
                      "tokens_prompt": n_ent, "tokens_por_s": round(n_novos / dt, 1) if dt > 0 else None})
        print(f"[{p['id']}] {dt:.2f}s {n_novos} tok | {p['prompt'][:60]!r}\n    -> {resp[:200]!r}")
    mem = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else None
    lat = [i["latencia_s"] for i in itens]
    tps = [i["tokens_por_s"] for i in itens if i["tokens_por_s"]]
    RESULTADOS["modelos"][nome] = {
        "model_id": mid, "dtype": str(dtype), "parametros": sum(p.numel() for p in m.parameters()),
        "tempo_carga_s": round(t_carga, 1), "pico_memoria_gpu_gb": round(mem, 2) if mem else None,
        "latencia_media_s": round(sum(lat) / len(lat), 3), "tokens_por_s_medio": round(sum(tps) / len(tps), 1),
        "itens": itens,
    }
    del m, tok; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

arq = os.path.join(PASTA, f"respostas_tucano2_{datetime.date.today()}.json")
with open(arq, "w", encoding="utf-8") as f:
    json.dump(RESULTADOS, f, ensure_ascii=False, indent=2)
print("\nSalvo em", arq)

In [ ]:
import pandas as pd
linhas = [{"modelo": n, "pico_gpu_gb": d["pico_memoria_gpu_gb"], "carga_s": d["tempo_carga_s"],
           "latencia_media_s": d["latencia_media_s"], "tokens/s": d["tokens_por_s_medio"], "params": d["parametros"]}
          for n, d in RESULTADOS["modelos"].items()]
pd.DataFrame(linhas)

## (Opcional) Velocidade na CPU, sem GPU

Serve para saber se o Tucano rodaria no notebook do dono (8 CPUs, 7 GB de RAM). **Para medir de verdade, troque o ambiente do Colab para CPU**
(Ambiente de execução > Alterar tipo > CPU) ou rode esta célula num Colab sem GPU. O Colab gratuito em CPU tem 2 vCPUs e ~12 GB de RAM:
o resultado vale como PISO; no notebook de 8 CPUs deve sair melhor (estimativa, não medida).
Em float32 o modelo de 0,5B ocupa ~2 GB e o de 1,5B ~6 GB (não cabe com folga em 7 GB); por isso a célula usa bfloat16 (metade da memória, mais lento em CPUs sem AVX512-BF16).

In [ ]:
import torch, time, os
from transformers import AutoTokenizer, AutoModelForCausalLM
torch.set_num_threads(os.cpu_count())
print("threads:", torch.get_num_threads(), "| GPU disponível:", torch.cuda.is_available())

def medir_cpu(model_id, dtype=torch.bfloat16, prompt="Explique em português o que é um JOIN em SQL.", novos=48):
    tok = AutoTokenizer.from_pretrained(model_id)
    m = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, device_map={"": "cpu"}).eval()
    msgs = [{"role": "user", "content": prompt}]
    ent = tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True), return_tensors="pt")
    with torch.no_grad():
        m.generate(**ent, max_new_tokens=4, do_sample=False)           # aquecimento
        t0 = time.time()
        out = m.generate(**ent, max_new_tokens=novos, do_sample=False, repetition_penalty=1.2)
    dt = time.time() - t0
    n = out.shape[1] - ent.input_ids.shape[1]
    print(f"{model_id} [{dtype}]: {n} tokens em {dt:.1f}s = {n/dt:.1f} tokens/s")
    del m
    return n / dt

for mid in ["Polygl0t/Tucano2-qwen-0.5B-Instruct"]:      # acrescente o 1.5B se a RAM do ambiente aguentar
    medir_cpu(mid)